# Lab 9 Inverse RL & Reward Inference  
### Adversarial Inverse Reinforcement Learning (AIRL) on **LunarLander-v2**

In this lab you will:

1. Train or load an **expert** policy with PPO (forward RL).
2. Collect **demonstrations** from the expert.
3. Train **AIRL** to infer a reward and learn a policy from demos.

Run on Google Colab:

[![Open In Collab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duke-trust-lab/intro_modern_rl/blob/main/lab9/lab9.ipynb)

---

## Learning objectives

By the end of this notebook, you should be able to:

- Explain the difference between **RL**, **IRL**, and **imitation learning**.
- Describe (at a high level) how **AIRL** uses an adversarial game to learn a reward.
- Run an AIRL pipeline end-to-end and evaluate whether the learned policy matches expert behavior.


### Concept refresher

**Forward RL:** reward is given, learn a policy.

- Policy:  π(a|s)
- Objective: maximize expected return  
  **J(π) = E[∑ γᵗ r(sₜ, aₜ)]**

**Inverse RL:** demonstrations are given, infer a reward (and usually a policy).

- Given expert trajectories τᴱ = (s₀,a₀,s₁,a₁,…)
- Find r̂(s,a) such that the optimal (or near-optimal) policy under r̂ explains the demonstrations.

**AIRL (Adversarial IRL):** trains a discriminator to distinguish expert transitions from policy transitions.
The discriminator structure corresponds to a learned reward + shaping term, encouraging a *reward that generalizes* beyond the expert’s exact state distribution.



In [15]:
# --- Original (Colab-only) install commands, kept for reference ---
# !pip -q uninstall -y gymnasium shimmy stable-baselines3 imitation box2d-py Box2D pygame
#
# !pip -q install -U pip setuptools wheel
# !apt-get -qq update
# !apt-get -qq install -y swig ffmpeg
#
# !pip -q install "gymnasium==0.29.1" "shimmy==1.3.0" "stable-baselines3==2.3.2" "imitation==1.0.0" "imageio[ffmpeg]"
#
# !pip -q install "Box2D" "pygame==2.5.2" --only-binary=:all:

# --- Windows/Colab-compatible version ---
import sys, platform

# Use %pip so packages install into THIS kernel's environment (not whatever `pip` is on PATH).
%pip -q uninstall -y gymnasium shimmy stable-baselines3 imitation box2d-py Box2D pygame

# apt-get only exists on Linux/Colab; on Windows, Box2D installs from a prebuilt wheel.
if platform.system() == "Linux":
    !apt-get -qq update
    !apt-get -qq install -y swig ffmpeg

%pip -q install "gymnasium==0.29.1" "shimmy==1.3.0" "stable-baselines3==2.3.2" "imitation==1.0.0" "imageio[ffmpeg]"

%pip -q install "Box2D" "pygame==2.5.2" --only-binary=:all:

print("Done. Restart the kernel (Kernel > Restart) before running the next cell.")


Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.
You can safely remove it manually.


Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rl-zoo3 2.9.1 requires shimmy~=2.0, but you have shimmy 1.3.0 which is incompatible.
sb3-contrib 2.9.0 requires stable_baselines3<3.0,>=2.9.0, but you have stable-baselines3 2.3.2 which is incompatible.

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Done. Restart the kernel (Kernel > Restart) before running the next cell.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import os, numpy as np, torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import imageio.v2 as imageio
import gymnasium as gym

from stable_baselines3 import PPO
from imitation.util.util import make_vec_env
from imitation.data import rollout
from imitation.data.wrappers import RolloutInfoWrapper
from imitation.algorithms.adversarial.airl import AIRL
from imitation.rewards.reward_nets import BasicShapedRewardNet

OUT_DIR = "outputs_lab11"
os.makedirs(OUT_DIR, exist_ok=True)


We’ll use:
- `evaluate_policy`: compute mean/std episodic return for a policy
- `record_video`: record an MP4 for qualitative inspection

> Qualitative inspection matters in IRL because “high reward” is not the only thing we care about ... we care about **matching behavior**


In [17]:
def evaluate_policy(model, env_id="LunarLander-v2", n_episodes=10, deterministic=True, max_steps=1000, seed=0):
    rewards = []
    for ep in range(n_episodes):
        env = gym.make(env_id)
        obs, _ = env.reset(seed=seed + ep)
        total = 0.0
        for _ in range(max_steps):
            action, _ = model.predict(obs, deterministic=deterministic)
            obs, r, terminated, truncated, _ = env.step(action)
            total += float(r)
            if terminated or truncated:
                break
        rewards.append(total)
        env.close()
    return float(np.mean(rewards)), float(np.std(rewards))


def record_video(model, env_id="LunarLander-v2", out_dir=OUT_DIR, name="policy_demo", fps=30, max_steps=1000):
    env = gym.make(env_id, render_mode="rgb_array")
    frames = []
    obs, _ = env.reset()
    total = 0.0
    for _ in range(max_steps):
        action, _ = model.predict(obs, deterministic=True)
        obs, r, terminated, truncated, _ = env.step(action)
        total += float(r)
        frames.append(env.render())
        if terminated or truncated:
            break
    env.close()
    path = os.path.join(out_dir, f"{name}.mp4")
    imageio.mimsave(path, frames, fps=fps, codec="libx264")
    print(f"Saved: {path} | Return: {total:.1f} | Steps: {len(frames)}")
    return path, total


### Train or load an Expert (Forward RL)

We first train an expert PPO policy on the environment’s **true reward**.  
Then AIRL will try to reproduce expert behavior **without directly using the reward** (only demos)


In [18]:
ENV_ID = "LunarLander-v2"
EXPERT_PATH = os.path.join(OUT_DIR, "expert_ppo.zip")

# PPO params

expert_cfg = dict(
    n_envs=8, #<--adjust as needed (16)
    total_timesteps=500_000, #<--adjust as needed (200_000)
    learning_rate=3e-4,
    n_steps=1024, #<--adjust as needed (2048)
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
)


In [19]:
def load_or_train_expert(env_id=ENV_ID, expert_path=EXPERT_PATH, seed=42, verbose=1, **cfg):
    rng = np.random.default_rng(seed=seed)
    env = make_vec_env(
        env_id,
        n_envs=cfg["n_envs"],
        rng=rng,
        post_wrappers=[lambda env, idx: RolloutInfoWrapper(env)]
    )

    if os.path.exists(expert_path):
        print("Loading existing expert:", expert_path)
        expert = PPO.load(expert_path, env=env)
    else:
        print("Training expert PPO")
        expert = PPO(
            "MlpPolicy",
            env,
            verbose=verbose,
            learning_rate=cfg["learning_rate"],
            n_steps=cfg["n_steps"],
            batch_size=cfg["batch_size"],
            n_epochs=cfg["n_epochs"],
            gamma=cfg["gamma"],
            gae_lambda=cfg["gae_lambda"],
            clip_range=cfg["clip_range"],
            ent_coef=cfg["ent_coef"],
            seed=seed,
        )
        expert.learn(total_timesteps=cfg["total_timesteps"])
        expert.save(expert_path)
        print("Saved expert to:", expert_path)

    mean_r, std_r = evaluate_policy(expert, env_id=env_id, n_episodes=10)
    print(f"Expert eval: {mean_r:.1f} ± {std_r:.1f} (10 eps)")

    return expert, env, mean_r, std_r

expert, vec_env, expert_mean, expert_std = load_or_train_expert(**expert_cfg)

Loading existing expert: outputs_lab11\expert_ppo.zip
Expert eval: 256.5 ± 19.3 (10 eps)


**Quick check:** record one expert episode for qualitative sanity.

If the expert is good, you’ll see controlled hovering and a safe landing (most of the time).


In [20]:
_ = record_video(expert, ENV_ID, OUT_DIR, name="expert_demo")


Saved: outputs_lab11\expert_demo.mp4 | Return: 277.7 | Steps: 285


---

### Collect expert demonstrations

AIRL trains using a dataset of transitions sampled from expert rollouts.  
We’ll collect `min_timesteps` transitions across multiple trajectories.

> For IRL, the **coverage** of demonstrations matters: if the expert never visits some region of the state space, the inferred reward may be unreliable there.


In [21]:
def collect_demonstrations(expert, env, min_timesteps=20_000, seed=42):
    rng = np.random.default_rng(seed=seed)
    rollouts_ = rollout.rollout(
        expert,
        env,
        rollout.make_sample_until(min_timesteps=min_timesteps),
        rng=rng
    )
    transitions_ = rollout.flatten_trajectories(rollouts_)
    print(f"Collected transitions: {len(transitions_)}")
    print(f"Number of trajectories: {len(rollouts_)}")
    return transitions_, rollouts_

MIN_DEMO_TIMESTEPS = 50_000

transitions, rollouts_raw = collect_demonstrations(expert, vec_env, min_timesteps=MIN_DEMO_TIMESTEPS)


Collected transitions: 53353
Number of trajectories: 153


### Train AIRL

AIRL components:

- **Generator / policy** (PPO): produces rollouts to improve.
- **Discriminator**: tries to tell expert transitions apart from generator transitions.
- **Reward network**: inferred reward used by the generator policy.

In this notebook we use a shaped reward network:
`BasicShapedRewardNet = r_θ(s,a) + γΦ(s') − Φ(s)`

That shaping term helps represent *equivalent rewards* that produce the same optimal behavior


In [22]:
def train_airl(
    transitions,
    env_id=ENV_ID,
    out_dir=OUT_DIR,
    load_existing=True,
    verbose=1,
    # env
    n_envs=8,
    env_seed=999,
    # reward net
    use_next_state=False,
    use_done=True,
    reward_hid_sizes=(256, 256),
    potential_hid_sizes=(256, 256),
    discount_factor=0.99,
    # generator PPO
    gen_learning_rate=3e-4,
    gen_n_steps=1024,
    gen_batch_size=64,
    gen_n_epochs=10,
    gen_gamma=0.99,
    gen_gae_lambda=0.95,
    gen_clip_range=0.2,
    gen_ent_coef=0.01,
    # discriminator
    disc_learning_rate=3e-4,
    n_disc_updates_per_round=4,
    demo_batch_size=512,
    # training
    gen_train_timesteps_multiplier=1.0,
    total_timesteps=300_000,
):
    # AIRL training environment
    env_airl = make_vec_env(
        env_id,
        n_envs=n_envs,
        rng=np.random.default_rng(seed=env_seed),
        post_wrappers=[lambda env, idx: RolloutInfoWrapper(env)]
    )

    reward_net = BasicShapedRewardNet(
        env_airl.observation_space,
        env_airl.action_space,
        use_state=True,
        use_action=True,
        use_next_state=use_next_state,
        use_done=use_done,
        reward_hid_sizes=list(reward_hid_sizes),
        potential_hid_sizes=list(potential_hid_sizes),
        discount_factor=discount_factor,
    )

    # gen_train_timesteps is how many env steps PPO runs per AIRL round
    gen_train_timesteps = int(n_envs * gen_n_steps * gen_train_timesteps_multiplier)

    learner = PPO(
        "MlpPolicy",
        env_airl,
        verbose=verbose,
        learning_rate=gen_learning_rate,
        n_steps=gen_n_steps,
        batch_size=gen_batch_size,
        n_epochs=gen_n_epochs,
        gamma=gen_gamma,
        gae_lambda=gen_gae_lambda,
        clip_range=gen_clip_range,
        ent_coef=gen_ent_coef,
    )

    policy_path = os.path.join(out_dir, "airl_policy.zip")
    reward_path = os.path.join(out_dir, "airl_reward_net.pt")

    if load_existing and os.path.exists(policy_path) and os.path.exists(reward_path):
        print("Loading existing AIRL artifacts")
        learner = PPO.load(policy_path, env=env_airl)
        reward_net.load_state_dict(torch.load(reward_path, map_location="cpu"))

    airl = AIRL(
        demonstrations=transitions,
        demo_batch_size=demo_batch_size,
        venv=env_airl,
        gen_algo=learner,
        reward_net=reward_net,
        allow_variable_horizon=True,
        disc_opt_kwargs={"lr": disc_learning_rate},
        n_disc_updates_per_round=n_disc_updates_per_round,
        gen_train_timesteps=gen_train_timesteps,
    )

    if not (load_existing and os.path.exists(policy_path) and os.path.exists(reward_path)):
        print(f"Training AIRL for {total_timesteps:,} timesteps")
        airl.train(total_timesteps=total_timesteps)
        print("Saving AIRL artifacts")
        airl.gen_algo.save(policy_path)
        torch.save(airl._reward_net.state_dict(), reward_path)

    mean_r, std_r = evaluate_policy(airl.gen_algo, env_id=env_id, n_episodes=20)
    print(f"AIRL eval: {mean_r:.1f} ± {std_r:.1f} (20 eps)")

    return airl, mean_r, std_r


In [23]:
airl_cfg_fast = dict(
    n_envs=4,
    reward_hid_sizes=(256, 256),
    potential_hid_sizes=(256, 256),
    demo_batch_size=512,
    disc_learning_rate=3e-4,
    n_disc_updates_per_round=4,
    gen_learning_rate=3e-4,
    gen_n_steps=1024,
    gen_gae_lambda=0.95,
    gen_ent_coef=0.01,
    gen_train_timesteps_multiplier=1.0,
    total_timesteps=300_000,
    load_existing=True,
    verbose=1
)

airl_cfg = dict(
    n_envs=8,
    reward_hid_sizes=(512, 512),
    potential_hid_sizes=(512, 512),
    demo_batch_size=1024,
    disc_learning_rate=4.9e-4,
    n_disc_updates_per_round=4,
    gen_learning_rate=6.35e-4,
    gen_n_steps=2048,
    gen_gae_lambda=0.949,
    gen_ent_coef=0.0488,
    gen_train_timesteps_multiplier=0.813,
    total_timesteps=800_000, #<- May want to reduce, takes ~20 mins to run on an A100 with 800k timesteps
    load_existing=True,
    verbose=1
)

airl, airl_mean, airl_std = train_airl(transitions, **airl_cfg)


Using cpu device
Loading existing AIRL artifacts
Running with `allow_variable_horizon` set to True. Some algorithms are biased towards shorter or longer episodes, which may significantly confound results. Additionally, even unbiased algorithms can exploit the information leak from the termination condition, producing spuriously high performance. See https://imitation.readthedocs.io/en/latest/getting-started/variable-horizon.html for more information.
AIRL eval: -52.5 ± 26.1 (20 eps)


### Compare performance

We’ll compare:
- Expert mean return
- AIRL mean return

### Checkpoint
If AIRL gets *high reward* but looks behaviorally different, is that “success”? Why or why not?

Ans: No, high reward does not always mean success in IRL, because the goal is to recover the expert's intent and match its behavior.

In [24]:
print(f"Expert: {expert_mean:.1f} ± {expert_std:.1f}")
print(f"AIRL:   {airl_mean:.1f} ± {airl_std:.1f}")


Expert: 256.5 ± 19.3
AIRL:   -52.5 ± 26.1


Record an AIRL episode.  
Compare the video with the expert: smoothness, hovering, leg contact timing, crash vs safe landings, etc.

Ans: AIRL gets low reward, and it does not succeed in landing yet. It lowers its altitude smoothly but eventually fly/hover around above the land rather than landing. No crash but its legs never touch down.

In [25]:
_ = record_video(airl.gen_algo, ENV_ID, OUT_DIR, name="airl_demo")


Saved: outputs_lab11\airl_demo.mp4 | Return: -72.5 | Steps: 1000


### Checkpoint

*Reflect on the following statements about IRL. How do your results highlight some of these challenges with IRL?*

1. IRL ≠ imitation → AIRL struggles because it must infer why actions were taken

2. High-quality demos are necessary but not sufficient

3. Reward learning is fragile

4. Small reward errors → catastrophic policy failure, especially in long-horizon control

Ans: 
1. In my run, IRL can lower altitude but still struggles and hovers around without landing, so it does not imitate. If it imitates, then it should copy the actions from experts and land successfully.
2. The expert demos seem to look good, but it does not guarantee IRL success. It may depend on several other factors.
3. IRL reward seems to be quite noisy. The original notebook was -30, my first run was -21, and when I re-ran the same baseline code and settings I got -52. Other seeds I tried gave -88 and -122 (some of them crashed instead of hovering). The same setup giving such different results indicates that reward learning is fragile.

In [13]:
# TODO
# How can you improve the performance of AIRL?

# ---------------------------------------------------------------------------
# Baseline (airl_cfg, 800k steps): Expert ~256 vs AIRL ~ -52 on the re-run (first run: -21);
# other seeds gave -88 and -122. Re-run policy hovers ~0.6 above the pad, never lands (20/20 timeouts).
# Training log (first run): true return never beat ~ -200; policy flips crash <-> hover;
# discriminator often wins (disc_acc_gen ~0.9+) -> generator can't keep up.
#
# Optuna search (10 trials, only 120k steps each, single seed, TPE ~ random search here)
# width | disc_lr | gen_lr  | ent_coef | n_envs | eval return
#   128 | 3.4e-5  | 3.5e-4  | 0.0060   | 8      | -113  <- best
#   128 | 4.3e-5  | 5.3e-5  | 0.0093   | 4      | -123
#   128 | 1.2e-5  | 3.7e-5  | 0.0016   | 8      | -389
#   256 | 1.4e-4  | 7.1e-4  | 0.0013   | 8      | -133
#   256 | 1.2e-4  | 7.0e-5  | 0.0125   | 8      | -144
#   512 | 3.2e-5  | 8.6e-5  | 0.0026   | 8      | -489
#   512 | 4.0e-4  | 8.4e-5  | 0.0212   | 4      | -556
#   512 | 1.4e-4  | 3.4e-5  | 0.0077   | 8      | -623
#   512 | 2.3e-4  | 5.2e-5  | 0.0055   | 2      | -1108
#   512 | 4.4e-4  | 1.6e-5  | 0.0265   | 4      | -1108
#
# Takeaways: (1) every 512-wide net was worst -> try 128/256; (2) disc_lr ~3e-5..1.5e-4
# (baseline 4.9e-4 is higher); (3) gen_lr ~3.5e-4; (4) ent_coef ~0.005-0.01 (weak evidence);
# (5) n_envs=8. Caveat: 120k steps is short and favors small nets; none beat the 800k baseline,
# so test these settings at a full budget before concluding anything.
# ---------------------------------------------------------------------------


### v3: lower the generator's entropy bonus

**Problem:** the baseline AIRL policy hovers above the pad (or sometimes crashes) and never lands, and its result changes a lot from run to run.

**Change in v3:** one parameter. `gen_ent_coef` goes from 0.0488 (baseline) to **0.01**. Everything else is the baseline `airl_cfg`. The learned reward, the discriminator and the evaluation are unchanged. There is no behavioral cloning and no reward normalization or shifting.

**Why I tried it (a hypothesis, not proven):** the entropy bonus rewards the policy for staying random. At 0.0488 it competes with the learned reward, which differs only slightly between states, so the policy is never pushed to commit to landing. A smaller bonus lets the learned reward matter more. A larger value did not help either (0.03 and 0.1 stayed in the hover or crash regime).

**Results** (800k steps, same 20 evaluation episodes for every run):

| Run | Seed | Mean return | Landed / crashed / timed out |
|---|---|---|---|
| Baseline (`airl_cfg`) | first run | -21.1 | 0 / 0 / 20 |
| Baseline | re-run | -52.5 | 0 / 0 / 20 |
| Baseline | 0, 1 | -121.9, -88.4 | 0 / 10 / 10 and 2 / 18 / 0 |
| **v3 (`gen_ent_coef=0.01`)** | 0 (this notebook) | **231.2** | 19 / 1 / 0 |
| v3 | 1 | 244.2 | 20 / 0 / 0 |
| v3 | 2 | 42.8 | 1 / 19 / 0 |
| v3 | 3 | 256.7 | 19 / 1 / 0 |

The expert scores 256.5. v3 lands in 3 of 4 seeds, with a mean of about 194 over the 4 seeds against about -71 for the 4 baseline runs, but it is still seed-sensitive (seed 2 crashes). Training 1.6M steps instead of 800k gave 248.2 on seed 0.

The artifacts for seed 0 are saved in `outputs_lab11/airl_v3`

In [28]:
# v3: lower the generator entropy bonus (gen_ent_coef 0.0488 -> 0.01); everything else is airl_cfg
airl_cfg_v3 = dict(airl_cfg, gen_ent_coef=0.01)
V3_DIR = os.path.join(OUT_DIR, "airl_v3")
os.makedirs(V3_DIR, exist_ok=True)

def evaluate_detail(model, env_id=ENV_ID, n_episodes=20, seed=0, max_steps=1000):
    """Like evaluate_policy, but also counts landings, crashes and timeouts."""
    landed = crashed = timeout = 0
    for ep in range(n_episodes):
        env = gym.make(env_id)
        obs, _ = env.reset(seed=seed + ep)
        term = trunc = False
        r = 0.0
        for _ in range(max_steps):
            action, _ = model.predict(obs, deterministic=True)
            obs, r, term, trunc, _ = env.step(action)
            if term or trunc:
                break
        env.close()
        if trunc or not term:
            timeout += 1
        elif r < -50:
            crashed += 1
        else:
            landed += 1
    return landed, crashed, timeout

airl_v3, v3_mean, v3_std = train_airl(transitions, out_dir=V3_DIR, **airl_cfg_v3)
landed, crashed, timeout = evaluate_detail(airl_v3.gen_algo)
print(f"Expert:    {expert_mean:.1f} ± {expert_std:.1f}")
print(f"AIRL base: {airl_mean:.1f} ± {airl_std:.1f}")
print(f"AIRL v3:   {v3_mean:.1f} ± {v3_std:.1f}  (landed {landed}, crashed {crashed}, timed out {timeout} of 20)")
_ = record_video(airl_v3.gen_algo, ENV_ID, V3_DIR, name="airl_v3_demo")

Using cpu device
Loading existing AIRL artifacts
Running with `allow_variable_horizon` set to True. Some algorithms are biased towards shorter or longer episodes, which may significantly confound results. Additionally, even unbiased algorithms can exploit the information leak from the termination condition, producing spuriously high performance. See https://imitation.readthedocs.io/en/latest/getting-started/variable-horizon.html for more information.
AIRL eval: 231.2 ± 64.8 (20 eps)
Expert:    256.5 ± 19.3
AIRL base: -52.5 ± 26.1
AIRL v3:   231.2 ± 64.8  (landed 19, crashed 1, timed out 0 of 20)


Saved: outputs_lab11\airl_v3\airl_v3_demo.mp4 | Return: 217.1 | Steps: 416


### Checkpoint

1. Why can multiple different rewards explain the same demonstrations?<br>
Ans: Demos only show behavior, and many rewards make the expert optimal. Demos also only constrain the reward on states the expert visited.<br>
2. If AIRL learned a reward that works near the expert’s states, why might it fail off-distribution?<br>
Ans: The expert never visits states like hovering or crashing, so the reward there is unconstrained. The policy drifts into those states and can exploit them.<br>
3. What evaluation would you add to judge *behavioral similarity* beyond episodic return?<br>
Ans: Landing success rate, and leg-contact timing.<br>
4. Can AIRL learn a reward that “looks right” on demos but induces unintended behavior? Give a concrete example in LunarLander<br>
Ans: Yes. A reward that favors the expert's low, centered, slow states can be earned forever by hovering just above the pad.<br>

## TODO: Hyperparameter search (Optuna)


In [12]:
# Uncomment to run a tiny Optuna search (can still take a while)
!pip -q install optuna

import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

# Optuna trials save airl_policy.zip / airl_reward_net.pt; use a separate folder so they
# don't overwrite the baseline AIRL artifacts in OUT_DIR.
OPTUNA_DIR = os.path.join(OUT_DIR, "optuna_tmp")
os.makedirs(OPTUNA_DIR, exist_ok=True)

def objective(trial):
    reward_width = trial.suggest_categorical("reward_width", [128, 256, 512])
    disc_lr = trial.suggest_float("disc_lr", 1e-5, 1e-3, log=True)
    gen_lr = trial.suggest_float("gen_lr", 1e-5, 1e-3, log=True)
    ent = trial.suggest_float("ent_coef", 0.001, 0.05, log=True)
    n_envs = trial.suggest_categorical("n_envs", [2, 4, 8])

    airl_tmp, mean_r, std_r = train_airl(
        transitions,
        n_envs=n_envs,
        reward_hid_sizes=(reward_width, reward_width),
        potential_hid_sizes=(reward_width, reward_width),
        disc_learning_rate=disc_lr,
        gen_learning_rate=gen_lr,
        gen_ent_coef=ent,
        total_timesteps=120_000,
        out_dir=OPTUNA_DIR,   # keep trial artifacts out of the baseline files in OUT_DIR
        load_existing=False,
        verbose=0
    )
    return mean_r

study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=0), pruner=MedianPruner())
study.optimize(objective, n_trials=10)

print("Best trial:", study.best_trial.number)
print("Best value:", study.best_value)
print("Best params:", study.best_params)



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip
c:\Users\beung-yoga\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-11-17 00:35:14,610] A new study created in memory with name: no-name-38d7df3d-1353-4ab2-adca-b98f5904f450


Running with `allow_variable_horizon` set to True. Some algorithms are biased towards shorter or longer episodes, which may significantly confound results. Additionally, even unbiased algorithms can exploit the information leak from the termination condition, producing spuriously high performance. See https://imitation.readthedocs.io/en/latest/getting-started/variable-horizon.html for more information.
Training AIRL for 120,000 timesteps


round:   0%|          | 0/14 [00:00<?, ?it/s]

------------------------------------------
| raw/                        |          |
|    gen/rollout/ep_len_mean  | 89.7     |
|    gen/rollout/ep_rew_mean  | -178     |
|    gen/time/fps             | 4097     |
|    gen/time/iterations      | 1        |
|    gen/time/time_elapsed    | 1        |
|    gen/time/total_timesteps | 8192     |
------------------------------------------
--------------------------------------------------
| raw/                                |          |
|    disc/disc_acc                    | 0.5      |
|    disc/disc_acc_expert             | 1        |
|    disc/disc_acc_gen                | 0        |
|    disc/disc_entropy                | 0.473    |
|    disc/disc_loss                   | 0.948    |
|    disc/disc_proportion_expert_pred | 1        |
|    disc/disc_proportion_expert_true | 0.5      |
|    disc/global_step                 | 1        |
|    disc/n_expert                    | 512      |
|    disc/n_generated                 | 512      |
-

round:   7%|▋         | 1/14 [00:06<01:21,  6.25s/it]

----------------------------------------------------
| raw/                               |             |
|    gen/rollout/ep_len_mean         | 86.3        |
|    gen/rollout/ep_rew_mean         | -169        |
|    gen/rollout/ep_rew_wrapped_mean | 8.35        |
|    gen/time/fps                    | 3982        |
|    gen/time/iterations             | 1           |
|    gen/time/time_elapsed           | 2           |
|    gen/time/total_timesteps        | 16384       |
|    gen/train/approx_kl             | 0.016476242 |
|    gen/train/clip_fraction         | 0.102       |
|    gen/train/clip_range            | 0.2         |
|    gen/train/entropy_loss          | -1.38       |
|    gen/train/explained_variance    | -0.365      |
|    gen/train/learning_rate         | 7.04e-05    |
|    gen/train/loss                  | 0.0268      |
|    gen/train/n_updates             | 10          |
|    gen/train/policy_gradient_loss  | -0.01       |
|    gen/train/value_loss            | 0.153  

... (490 outputs left out)

round: 100%|██████████| 14/14 [01:28<00:00,  6.35s/it]

Saving AIRL artifacts



[I 2026-11-17 00:55:11,074] Trial 9 finished with value: -623.1898695754217 and parameters: {'reward_width': 512, 'disc_lr': 0.00013589482177682005, 'gen_lr': 3.394524768362828e-05, 'ent_coef': 0.007744310329461799, 'n_envs': 8}. Best is trial 3 with value: -113.24496260657868.


AIRL eval: -623.2 ± 160.3 (20 eps)
Best trial: 3
Best value: -113.24496260657868
Best params: {'reward_width': 128, 'disc_lr': 3.381514267820288e-05, 'gen_lr': 0.0003535634629148812, 'ent_coef': 0.005956425742609303, 'n_envs': 8}


### Checkpoint

- What are we optimizing in AIRL (at a high level)?<br>
Ans: AIRL optimizes a game. The discriminator tells expert transitions from the policy's, and the policy maximizes the learned reward to fool it. At the optimum the policy's behavior matches the expert's.<br>
- What does it mean for IRL to “succeed” beyond having a high episodic return?<br>
Ans: Success also means matching behavior (landing, smooth descent), recovering a reward that works under changed conditions.<br>